# 时序记忆（Temporal Memory）

> **给你的智能体时间感：记录事情发生的时间，在关键时优先使用最新的信息。**

想象一本笔记本，每条记录的页边都写着日期。有人问「项目进展如何？」时，你不会从头读到尾，而是直接翻到最近的一页。旧记录可能仍然重要，但你会把最新的看得更重。

标准的基于向量的检索（按语义相似度搜索）把所有记忆当作同样「新」。你问智能体「项目进展如何？」，它可能把三个月前的更新和昨天的更新一起返回，而它没有任何信号表明旧的那条已经过时。在真实任务里，事情是*何时*被记录的，往往和记录了*什么*同样重要。

时序记忆填补了这个空缺。它给每条记忆挂上结构化的时间戳，并在检索流程中加入时间感知。核心想法来自认知科学。Endel Tulving 关于情景记忆（episodic memory，我们对亲身经历的记忆）的研究表明，人类的回忆天然带有时间戳。我们记住的不只是事实，还有自己*在某个特定时刻*经历过它。

Generative Agents 论文（Park et al., 2023）把这个洞见付诸实践。它用一个新近度（recency）分数为记忆检索加权。这让模拟智能体自然偏向最近的事件，同时重要的旧记忆仍然有机会浮现。

在本 notebook 中，你将从零构建一个时序记忆系统。你会实现衰减函数、时间范围过滤、新近度加权检索和时间线构建。学完之后，你的智能体不仅知道*发生了什么*，还知道*什么时候*发生的。

## 关键概念

- **时间戳元数据（timestamp metadata）**：每条记忆都带有结构化的时间字段。`created_at` 记录记忆的存储时间，`event_time` 记录所描述事件的实际发生时间，`last_accessed` 记录最近一次被检索的时间。这些字段支撑过滤、排序和衰减计算。

- **时间衰减函数（temporal decay function）**：一种数学函数，随记忆变旧而降低它的有效得分。常见选择有指数衰减（`score *= e^(-lambda * age)`）和线性衰减。衰减率决定旧记忆多快失去影响力。

- **半衰期（half-life）**：记忆的新近度得分降到 50% 所需的时间。半衰期为 7 天，意味着一条一周前的记忆在新近度上只有全新记忆的一半。这是你可以根据领域调整的参数。

- **新近度评分（recency scoring）**：一种检索时的计算，把语义相关性（含义匹配程度）和时间新鲜度（记忆有多新）融合在一起。当两条记忆与查询匹配得一样好时，较新的那条排名更高。

- **时间范围查询（time-range query）**：把检索限制在一个特定的时间窗口内。例如「过去 24 小时的对话」或「3 月 1 日到 3 月 15 日之间的事件」。

- **时点检索（"as of" retrieval）**：重建智能体在过去某个时间点知道什么。你会排除那个时间点之后创建的所有记忆。这对审计和调试很有用。

- **事件时间线（event timeline）**：一组按时间排序的相关记忆，串成一个按时间推进的故事。它帮助智能体概括局势是如何随时间发展的。

## 架构

<p align="center">
  <img src="../../images/diagrams/18_temporal_memory.svg" alt="时序记忆架构图" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart LR
    subgraph Storage
        MS[(Memory Store<br/>with timestamps)]
    end

    Q["Query + optional<br/>time constraints"] --> TRF[Time Range<br/>Filter]
    MS --> TRF
    TRF --> SS[Semantic<br/>Scorer]
    TRF --> TS[Temporal Scorer<br/>decay function]
    SS --> CR[Combined<br/>Ranking]
    TS --> CR
    CR --> TopK[Top-K<br/>Results]

    MS --> TL[Timeline<br/>Constructor]
    TL --> ET[Event<br/>Timeline]

    style MS fill:#4a9eff,color:#fff
    style TS fill:#ff6b6b,color:#fff
    style CR fill:#51cf66,color:#fff
```

</details>

**数据流：** 查询带着可选的时间约束到来。时间范围过滤器（Time Range Filter）从记忆库中缩小候选集。两个评分器并行工作：语义评分器（Semantic Scorer）计算语义相似度，时间评分器（Temporal Scorer）用衰减函数计算新近度。综合排序（Combined Ranking）把两个得分合并（加权求和），返回 top-K 结果。另一条路径通向时间线构建器（Timeline Constructor），它把相关记忆按时间先后排序，支持叙事式检索。

## 环境准备

安装依赖并配置 API 访问。我们用 OpenAI SDK 做嵌入和聊天补全。

In [ ]:
%pip install -q python-dotenv numpy deepagents langchain-openai

导入所有需要的库。API 密钥从 `.env` 文件或环境变量加载。

In [ ]:
import os
import json
import numpy as np
from datetime import datetime, timedelta
from dataclasses import dataclass, field
from typing import Optional

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)
def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


print("✓ Model ready")


网关没有 `/embeddings` 接口时，用这份本地确定性向量兜底：词加字符 n-gram 哈希后归一化。配置 `LLM_EMBEDDING_BASE_URL` 和 `LLM_EMBEDDING_MODEL_NAME` 后，自动切换到真实向量端点。


In [ ]:
EMBEDDING_MODEL = os.getenv("LLM_EMBEDDING_MODEL_NAME", "")


def embed_texts(texts: list[str]) -> list[list[float]]:
    """把文本转成向量。

    当前网关没有 /embeddings 接口。默认退回到本地确定性向量：
    词 + 字符 n-gram 哈希后归一化，语义相近的文本仍会得到相近向量，
    足够演示记忆检索的几何逻辑。配置 LLM_EMBEDDING_BASE_URL 和
    LLM_EMBEDDING_MODEL_NAME 后，自动改用真实向量端点。
    """
    if EMBEDDING_MODEL:
        from openai import OpenAI

        client = OpenAI(
            api_key=os.getenv("LLM_EMBEDDING_API_KEY", os.getenv("LLM_API_KEY")),
            base_url=os.getenv("LLM_EMBEDDING_BASE_URL", os.getenv("LLM_BASE_URL")),
        )
        response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
        return [item.embedding for item in response.data]

    import hashlib
    import math
    import re as re_mod

    dim = 512
    vectors = []
    for text in texts:
        vec = [0.0] * dim
        for word in re_mod.findall(r"\w+", text.lower()):
            grams = {word} | {word[i:i + 3] for i in range(max(1, len(word) - 2))}
            for gram in grams:
                digest = hashlib.blake2b(gram.encode("utf-8"), digest_size=8).digest()
                index = int.from_bytes(digest, "big") % dim
                vec[index] += 1.0
        norm = math.sqrt(sum(v * v for v in vec)) or 1.0
        vectors.append([v / norm for v in vec])
    return vectors


## 实现

我们将构建四个组件：
1. 一个 `TemporalMemory` 数据类（dataclass），把内容和时间戳元数据存在一起。
2. 一个 `TemporalDecay` 类，提供可配置的衰减函数（指数和线性）。
3. 一个 `TemporalMemoryStore`，负责存储、综合评分检索、时间范围查询和时点过滤。
4. 一个时间线构建器，把相关记忆按时间先后排序。

### 嵌入辅助函数

我们需要一个函数把文本转换成嵌入（embedding，一组捕捉语义的数字）。含义相近的两段文本会得到相近的嵌入。我们用余弦相似度（cosine similarity，衡量两个向量之间的夹角）来比较它们。

In [ ]:
def get_embedding(text: str) -> list[float]:
    """获取给定文本的嵌入向量。"""
    return embed_texts([text])[0]


def cosine_similarity(a: list[float], b: list[float]) -> float:
    """计算两个向量之间的余弦相似度。

    返回值在 -1 到 1 之间，越大越相似。
    """
    a_arr = np.array(a)
    b_arr = np.array(b)
    dot = np.dot(a_arr, b_arr)
    norm = np.linalg.norm(a_arr) * np.linalg.norm(b_arr)
    if norm == 0:
        return 0.0
    return float(dot / norm)

### TemporalMemory 数据模型

每条记忆条目携带三个时间戳。`created_at` 是我们的存储时间，`event_time` 是所描述事件实际发生的时间（可能与存储时间不同），`last_accessed` 记录最近一次检索。我们还存一个基于内容和创建时间生成的唯一 ID。

In [ ]:
@dataclass
class TemporalMemory:
    """带时间元数据的单条记忆。"""
    content: str
    embedding: list[float]
    created_at: datetime
    event_time: Optional[datetime] = None
    last_accessed: Optional[datetime] = None
    metadata: dict = field(default_factory=dict)
    memory_id: str = ""

    def __post_init__(self):
        if not self.event_time:
            self.event_time = self.created_at
        if not self.last_accessed:
            self.last_accessed = self.created_at
        if not self.memory_id:
            # 由内容 + 时间戳生成确定性 ID
            raw = f"{self.content}:{self.created_at.isoformat()}"
            self.memory_id = hashlib.sha256(raw.encode()).hexdigest()[:12]

    def age_hours(self, now: datetime) -> float:
        """这条记忆创建后经过的小时数。"""
        return (now - self.created_at).total_seconds() / 3600

    def __repr__(self) -> str:
        preview = self.content[:60] + ("..." if len(self.content) > 60 else "")
        return f"TemporalMemory(id={self.memory_id}, content='{preview}')"

### 时间衰减函数

把衰减想象成面包的新鲜度。今天早上烤的面包正处于最佳状态，昨天的面包还不错，上周的面包已经不新鲜了。新鲜度下降得有多快，取决于你选的衰减函数。

**指数衰减（exponential decay）**一开始掉得快，之后越来越慢。你用一个半衰期来参数化它：得分降到 0.5 所需的时间。这与心理学中的艾宾浩斯遗忘曲线（Ebbinghaus forgetting curve）吻合。

**线性衰减（linear decay）**以恒定速率下降，直到归零。它更简单，也更容易推理。

In [ ]:
class TemporalDecay:
    """用可配置的衰减函数计算新近度得分。"""

    def __init__(self, decay_type: str = "exponential", half_life_hours: float = 168.0):
        """
        参数：
            decay_type: "exponential"（指数）或 "linear"（线性）
            half_life_hours: 得分降到 0.5 所需的时间（小时）。
                             默认 168，即 7 天。
        """
        self.decay_type = decay_type
        self.half_life_hours = half_life_hours
        # 指数衰减：lambda = ln(2) / half_life
        self.lambda_ = math.log(2) / half_life_hours

    def score(self, memory: TemporalMemory, now: datetime) -> float:
        """返回 0.0 到 1.0 之间的新近度得分。

        1.0 表示全新，0.0 表示完全衰减。
        """
        age_h = memory.age_hours(now)
        if age_h <= 0:
            return 1.0

        if self.decay_type == "exponential":
            return math.exp(-self.lambda_ * age_h)
        elif self.decay_type == "linear":
            # 在 2 倍半衰期处降到 0
            max_age = 2.0 * self.half_life_hours
            return max(0.0, 1.0 - (age_h / max_age))
        else:
            return 1.0  # 不衰减

    def __repr__(self) -> str:
        return f"TemporalDecay(type={self.decay_type}, half_life={self.half_life_hours}h)"

### 时序记忆库

这是核心类。它存储记忆，并用语义与时间相结合的评分来检索。关键方法：

- `add()`：存储一条新记忆，附带嵌入和时间戳。
- `query()`：检索最相关的记忆，融合语义相似度与新近度。
- `query_time_range()`：检索特定时间窗口内的记忆。
- `query_as_of()`：重建智能体在过去某个时刻知道什么。
- `build_timeline()`：把相关记忆按时间先后排序。

In [ ]:
class TemporalMemoryStore:
    """带时间感知检索的记忆库。"""

    def __init__(
        self,
        decay: Optional[TemporalDecay] = None,
        recency_weight: float = 0.3,
    ):
        """
        参数：
            decay: 用于新近度得分的衰减函数。默认为半衰期
                   7 天的指数衰减。
            recency_weight: 新近度相对语义相似度的权重。
                            0.0 表示纯语义，1.0 表示纯新近度。
        """
        self.memories: list[TemporalMemory] = []
        self.decay = decay or TemporalDecay()
        self.recency_weight = recency_weight

接下来我们添加用于存储新记忆的 `add` 方法，以及三个内部评分辅助方法。`_semantic_scores` 计算语义相似度，`_recency_scores` 计算时间衰减，`_combined_scores` 把两者混合成一个排序得分。

In [ ]:

def add(
    self,
    content: str,
    created_at: Optional[datetime] = None,
    event_time: Optional[datetime] = None,
    metadata: Optional[dict] = None,
) -> TemporalMemory:
    """存储一条新记忆，附带嵌入和时间戳。"""
    now = created_at or datetime.utcnow()
    embedding = get_embedding(content)
    mem = TemporalMemory(
        content=content,
        embedding=embedding,
        created_at=now,
        event_time=event_time or now,
        metadata=metadata or {},
    )
    self.memories.append(mem)
    return mem
TemporalMemoryStore.add = add

def _semantic_scores(
    self, query_embedding: list[float], candidates: list[TemporalMemory]
) -> list[float]:
    """计算查询与每个候选记忆之间的余弦相似度。"""
    return [cosine_similarity(query_embedding, m.embedding) for m in candidates]
TemporalMemoryStore._semantic_scores = _semantic_scores

def _recency_scores(
    self, candidates: list[TemporalMemory], now: datetime
) -> list[float]:
    """计算每个候选记忆的时间衰减得分。"""
    return [self.decay.score(m, now) for m in candidates]
TemporalMemoryStore._recency_scores = _recency_scores

def _combined_scores(
    self,
    semantic: list[float],
    recency: list[float],
    weight: Optional[float] = None,
) -> list[float]:
    """混合语义得分与新近度得分。

    final = (1 - w) * semantic + w * recency
    """
    w = weight if weight is not None else self.recency_weight
    return [(1 - w) * s + w * r for s, r in zip(semantic, recency)]
TemporalMemoryStore._combined_scores = _combined_scores

`query` 方法是主要的检索入口。它为每条记忆同时计算语义相似度和时间新近度，然后把两者混合成综合得分：`(1 - w) * semantic + w * recency`。既匹配查询含义、又足够新的记忆排名最高。

In [ ]:

def query(
    self,
    query_text: str,
    top_k: int = 5,
    now: Optional[datetime] = None,
    recency_weight: Optional[float] = None,
) -> list[dict]:
    """按语义 + 时间的综合得分检索记忆。

    返回一个字典列表，包含记忆、各项得分和排名。
    """
    if not self.memories:
        return []

    now = now or datetime.utcnow()
    query_emb = get_embedding(query_text)

    semantic = self._semantic_scores(query_emb, self.memories)
    recency = self._recency_scores(self.memories, now)
    combined = self._combined_scores(semantic, recency, recency_weight)

    # 按综合得分降序排列
    ranked = sorted(
        zip(self.memories, semantic, recency, combined),
        key=lambda x: x[3],
        reverse=True,
    )

    results = []
    for mem, sem_s, rec_s, comb_s in ranked[:top_k]:
        mem.last_accessed = now
        results.append({
            "memory": mem,
            "semantic_score": round(sem_s, 4),
            "recency_score": round(rec_s, 4),
            "combined_score": round(comb_s, 4),
        })
    return results
TemporalMemoryStore.query = query

两个专用查询方法处理带时间约束的检索。`query_time_range` 把记忆过滤到特定窗口（比如「本周」）。`query_as_of` 通过排除较新的记忆，重建智能体在过去某个时间点知道什么。

In [ ]:

def query_time_range(
    self,
    query_text: str,
    time_start: datetime,
    time_end: datetime,
    top_k: int = 5,
) -> list[dict]:
    """检索某个时间窗口内的记忆。

    先按 event_time 过滤，再按语义相似度排序。
    """
    query_emb = get_embedding(query_text)

    candidates = [
        m for m in self.memories
        if time_start <= m.event_time <= time_end
    ]
    if not candidates:
        return []

    scores = self._semantic_scores(query_emb, candidates)
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)

    return [
        {"memory": m, "semantic_score": round(s, 4)}
        for m, s in ranked[:top_k]
    ]
TemporalMemoryStore.query_time_range = query_time_range

`query_as_of` 重建智能体在特定过去时刻的知识。它排除在给定时间戳之后创建的记忆。这对审计很有用：「两周前智能体知道什么？」

In [ ]:

def query_as_of(
    self,
    query_text: str,
    as_of: datetime,
    top_k: int = 5,
) -> list[dict]:
    """把当前时间当作 `as_of` 来检索记忆。

    排除所有在 as_of 时间戳之后创建的记忆。
    用 as_of 作为新近度得分的参考时间。
    """
    query_emb = get_embedding(query_text)

    candidates = [m for m in self.memories if m.created_at <= as_of]
    if not candidates:
        return []

    semantic = self._semantic_scores(query_emb, candidates)
    recency = self._recency_scores(candidates, as_of)
    combined = self._combined_scores(semantic, recency)

    ranked = sorted(
        zip(candidates, semantic, recency, combined),
        key=lambda x: x[3],
        reverse=True,
    )

    return [
        {
            "memory": m,
            "semantic_score": round(s, 4),
            "recency_score": round(r, 4),
            "combined_score": round(c, 4),
        }
        for m, s, r, c in ranked[:top_k]
    ]
TemporalMemoryStore.query_as_of = query_as_of

`build_timeline` 找出与某个主题相关的记忆，并按时间先后排序。这对构建叙事很有用：「项目 X 在过去一个月是如何演进的？」

In [ ]:

def build_timeline(
    self,
    topic: str,
    time_start: Optional[datetime] = None,
    time_end: Optional[datetime] = None,
    similarity_threshold: float = 0.3,
) -> list[TemporalMemory]:
    """为主题相关记忆构建按时间排序的时间线。

    先取出与主题的语义相似度超过阈值的记忆，
    再按 event_time 排序。
    """
    topic_emb = get_embedding(topic)
    scores = self._semantic_scores(topic_emb, self.memories)

    related = []
    for mem, score in zip(self.memories, scores):
        if score < similarity_threshold:
            continue
        if time_start and mem.event_time < time_start:
            continue
        if time_end and mem.event_time > time_end:
            continue
        related.append(mem)

    return sorted(related, key=lambda m: m.event_time)
TemporalMemoryStore.build_timeline = build_timeline

def __len__(self) -> int:
    return len(self.memories)
TemporalMemoryStore.__len__ = __len__

def __repr__(self) -> str:
    return f"TemporalMemoryStore({len(self.memories)} memories)"
TemporalMemoryStore.__repr__ = __repr__

### LLM 时间线摘要

拿到有序的时间线之后，我们可以让 LLM 生成一段叙事式摘要。这把一组原始记忆条目变成一个连贯的故事，讲述事件是如何展开的。

In [ ]:
def summarize_timeline(memories: list[TemporalMemory], topic: str) -> str:
    """用 LLM 为记忆时间线生成叙事式摘要。"""
    if not memories:
        return "No memories found for this topic."

    entries = []
    for m in memories:
        timestamp = m.event_time.strftime("%Y-%m-%d %H:%M")
        entries.append(f"[{timestamp}] {m.content}")

    timeline_text = "\n".join(entries)

    response = llm.invoke([
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant. Summarize the following "
                    "timeline of events into a brief narrative. Focus on how "
                    "the situation evolved over time. Use 3-5 sentences."
                ),
            },
            {
                "role": "user",
                "content": f"Topic: {topic}\n\nTimeline:\n{timeline_text}",
            },
        ])
    return to_text(response.content)

## 示例运行

想象一个智能体在数周内跟踪一个软件项目。团队成员在不同时间发布动态。智能体需要回答关于项目的问题，同时优先使用最新的信息。

我们创建时间戳分布在过去一个月内的记忆，然后测试全部四种检索模式：标准查询、时间范围查询、时点查询和时间线。

### 填充记忆库

我们创建一个使用指数衰减（半衰期 7 天）的记忆库，并添加横跨过去 30 天的项目动态。

In [ ]:
# 固定 "now"，保证示例可复现
NOW = datetime(2025, 4, 15, 10, 0, 0)

store = TemporalMemoryStore(
    decay=TemporalDecay(decay_type="exponential", half_life_hours=168),  # 半衰期 7 天
    recency_weight=0.3,
)


我们定义 10 条横跨过去 30 天的项目动态。每条记录发生了什么，以及发生在多少天前。循环把 `days_ago` 换算成实际时间戳，并存储每条动态。

In [ ]:
# 过去一个月的项目动态
updates = [
    {
        "content": "Project Alpha kicked off. The team decided to build a REST API "
                   "for the new inventory system. Tech stack: Python, FastAPI, PostgreSQL.",
        "days_ago": 30,
    },
    {
        "content": "Database schema design is complete. We have 12 tables covering "
                   "products, orders, customers, and inventory levels.",
        "days_ago": 25,
    },
    {
        "content": "The authentication module is done. We are using JWT tokens with "
                   "24-hour expiry and refresh token rotation.",
        "days_ago": 20,
    },
    {
        "content": "Performance testing revealed the product search endpoint is slow. "
                   "P95 latency is 800ms. The team is investigating indexing strategies.",
        "days_ago": 14,
    },
    {
        "content": "Added a composite index on (category, price, created_at) to the "
                   "products table. Product search P95 latency dropped from 800ms to 120ms.",
        "days_ago": 10,
    },
]

这里是剩下的项目动态：sprint 评审、通知功能上线、一个 bug 报告、bug 修复，以及最后一次状态会议。我们把两批合并，全部存入。

In [ ]:
more_updates = [
    {
        "content": "Sprint review: 85% of planned features are complete. Remaining work "
                   "includes order tracking notifications and admin dashboard.",
        "days_ago": 7,
    },
    {
        "content": "The order tracking notification system is now live. Customers receive "
                   "email and SMS updates when their order status changes.",
        "days_ago": 4,
    },
    {
        "content": "Bug report: some customers receive duplicate notification emails. "
                   "The team is investigating a race condition in the email queue.",
        "days_ago": 2,
    },
    {
        "content": "Fixed the duplicate notification bug. Root cause was a missing "
                   "idempotency key in the email queue consumer. Deployed the fix to production.",
        "days_ago": 1,
    },
    {
        "content": "Project Alpha status meeting: the admin dashboard is 60% complete. "
                   "Launch date is confirmed for May 1st. All critical bugs are resolved.",
        "days_ago": 0,
    },
]


updates.extend(more_updates)

for update in updates:
    event_time = NOW - timedelta(days=update["days_ago"])
    store.add(
        content=update["content"],
        created_at=event_time,
        event_time=event_time,
    )

print(f"Stored {len(store)} memories spanning {updates[0]['days_ago']} days.")

### 带新近度加权的标准查询

我们询问项目状态。开启新近度加权后，智能体应该优先采用最近的状态更新，而不是启动会议的记录，尽管两者在语义上都相关。

In [ ]:
results = store.query("What is the current project status?", top_k=5, now=NOW)

print("Query: 'What is the current project status?'\n")
print(f"{'Rank':<6} {'Combined':<10} {'Semantic':<10} {'Recency':<10} {'Age (days)':<12} Content")
print("-" * 100)

for i, r in enumerate(results, 1):
    mem = r["memory"]
    age_days = (NOW - mem.event_time).days
    preview = mem.content[:70] + ("..." if len(mem.content) > 70 else "")
    print(
        f"{i:<6} {r['combined_score']:<10.4f} {r['semantic_score']:<10.4f} "
        f"{r['recency_score']:<10.4f} {age_days:<12} {preview}"
    )

注意最近的状态会议（0 天前）排名第一，sprint 评审（7 天前）也排名不错。最初的启动会（30 天前）语义相关性很高，但新近度得分把它拖了下来。

接下来看看完全关闭新近度加权会发生什么。

In [ ]:
results_no_recency = store.query(
    "What is the current project status?",
    top_k=5,
    now=NOW,
    recency_weight=0.0,  # 纯语义
)

print("Query with recency_weight=0.0 (pure semantic):\n")
print(f"{'Rank':<6} {'Semantic':<10} {'Age (days)':<12} Content")
print("-" * 80)

for i, r in enumerate(results_no_recency, 1):
    mem = r["memory"]
    age_days = (NOW - mem.event_time).days
    preview = mem.content[:70] + ("..." if len(mem.content) > 70 else "")
    print(f"{i:<6} {r['semantic_score']:<10.4f} {age_days:<12} {preview}")

print("\nWithout recency weighting, stale updates may outrank current ones.")

### 时间范围查询

只检索过去一周的记忆。当用户问「最近发生了什么？」或「这个 sprint 有什么变化？」时很有用。

In [ ]:
week_start = NOW - timedelta(days=7)

results_week = store.query_time_range(
    query_text="project progress and issues",
    time_start=week_start,
    time_end=NOW,
    top_k=5,
)

print(f"Memories from the past 7 days (since {week_start.date()}):\n")
for i, r in enumerate(results_week, 1):
    mem = r["memory"]
    date_str = mem.event_time.strftime("%Y-%m-%d")
    preview = mem.content[:90] + ("..." if len(mem.content) > 90 else "")
    print(f"  {i}. [{date_str}] {preview}")

### 时点查询（As-Of Query）

重建智能体在某个过去日期知道什么。这对审计（「两周前智能体认为状态是什么？」）或调试很有用。

In [ ]:
# 14 天前智能体知道什么？
as_of_date = NOW - timedelta(days=14)

results_as_of = store.query_as_of(
    query_text="What is the project status?",
    as_of=as_of_date,
    top_k=3,
)

print(f"'As of' query: what did we know on {as_of_date.date()}?\n")
for i, r in enumerate(results_as_of, 1):
    mem = r["memory"]
    date_str = mem.event_time.strftime("%Y-%m-%d")
    print(f"  {i}. [{date_str}] (combined={r['combined_score']:.4f})")
    preview = mem.content[:100] + ("..." if len(mem.content) > 100 else "")
    print(f"     {preview}")
    print()

print("Notice: no memories from after the as_of date appear.")

### 时间线构建

构建一条按时间排序的时间线，展示通知系统的演进过程。记忆库会找出相关记忆，并按事件时间排序。

In [ ]:
timeline = store.build_timeline(
    topic="order notifications and email issues",
    similarity_threshold=0.25,
)

print(f"Timeline: 'order notifications and email issues' ({len(timeline)} entries)\n")
for mem in timeline:
    date_str = mem.event_time.strftime("%Y-%m-%d")
    print(f"  [{date_str}] {mem.content}")
    print()

### LLM 时间线摘要

把时间线交给 LLM，得到一段关于通知系统如何演进的叙事式摘要。

In [ ]:
summary = summarize_timeline(timeline, topic="order notification system")
print("Timeline Summary:\n")
print(summary)

### 对比衰减函数

我们把指数衰减和线性衰减的行为差异可视化。这有助于你为自己的场景选择合适的衰减函数。

In [ ]:
# 对比指数衰减与线性衰减（30 天内）
exp_decay = TemporalDecay(decay_type="exponential", half_life_hours=168)
lin_decay = TemporalDecay(decay_type="linear", half_life_hours=168)

hours_range = np.linspace(0, 720, 200)  # 0 到 30 天
dummy_emb = [0.0] * EMBED_DIM

exp_scores = []
lin_scores = []

for h in hours_range:
    dummy_mem = TemporalMemory(
        content="test",
        embedding=dummy_emb,
        created_at=NOW - timedelta(hours=float(h)),
    )
    exp_scores.append(exp_decay.score(dummy_mem, NOW))
    lin_scores.append(lin_decay.score(dummy_mem, NOW))

days_range = hours_range / 24

print("Decay score comparison (exponential vs linear):\n")
print(f"{'Age (days)':<14} {'Exponential':<14} {'Linear':<14}")
print("-" * 42)
for day_mark in [0, 1, 3, 7, 14, 21, 30]:
    idx = int(day_mark * 24 / 720 * 199)
    if idx >= len(exp_scores):
        idx = len(exp_scores) - 1
    print(f"{day_mark:<14} {exp_scores[idx]:<14.4f} {lin_scores[idx]:<14.4f}")

print("\nExponential decay drops fast initially then levels off.")
print("Linear decay drops at a constant rate and hits zero at 14 days (2x half-life).")

### 在聊天智能体中使用时序记忆

我们把时序记忆库接入一个对话式智能体。智能体在回答之前先检索相关记忆，它的回答会体现出记忆系统的时间感知。

In [ ]:
def temporal_agent_respond(
    user_question: str,
    memory_store: TemporalMemoryStore,
    now: datetime,
) -> str:
    """用时序记忆回答问题的智能体。"""
    # 按新近度加权检索前 5 条记忆
    results = memory_store.query(user_question, top_k=5, now=now)

    # 为 LLM 组织上下文
    context_entries = []
    for r in results:
        mem = r["memory"]
        age_days = (now - mem.event_time).days
        date_str = mem.event_time.strftime("%Y-%m-%d")
        context_entries.append(
            f"[{date_str}, {age_days} days ago] {mem.content}"
        )
    context_block = "\n\n".join(context_entries)

    response = llm.invoke([
            {
                "role": "system",
                "content": (
                    "You are a project assistant with access to timestamped "
                    "memories. Use the provided context to answer the user's "
                    "question. Prefer recent information when relevant. "
                    "Mention dates when they help clarify the answer. "
                    "Keep your response to 3-5 sentences."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Retrieved memories (most relevant first):\n\n"
                    f"{context_block}\n\n"
                    f"Question: {user_question}"
                ),
            },
        ])
    return to_text(response.content)


# 用几个问题测试智能体
questions = [
    "What is the current status of Project Alpha?",
    "Were there any bugs recently? What happened?",
    "How has the product search performance changed over time?",
]

for q in questions:
    print(f"User: {q}")
    answer = temporal_agent_respond(q, store, NOW)
    print(f"Agent: {answer}\n")

## 权衡

### 时序记忆的适用场景

- **动态环境**：信息会定期过时的场景（项目动态、市场数据、新闻）。衰减函数自然地降低过时记忆的优先级。
- **按时间推理**：智能体需要解释局势如何演进时。时间线构建器能生成有序的叙事，这是扁平的向量库做不到的。
- **审计与调试**：时点查询让你重建智能体在过去任意时刻的知识状态。这对理解智能体为什么给出某个回答很有价值。

### 它的不足

- **常青知识**：长期成立的事实（数学公式、公司政策、API 文档）会被衰减不公平地惩罚。你要么把某些记忆排除在衰减之外，要么把半衰期设得很长。
- **调参复杂**：选对半衰期和新近度权重需要反复实验。半衰期太短会忘掉重要上下文，太长又让时间感知失去意义。
- **存储开销**：每条记忆现在携带多个时间戳字段，查询时还要做衰减计算。对于大规模存储（10 万条以上），逐查询评分比纯向量检索增加延迟。
- **依赖时钟**：系统假设时间戳可靠。如果记忆乱序到达或时间戳有误，衰减评分会产生误导性的结果。

## 延伸阅读

- Park, J. S., et al. (2023). ["Generative Agents: Interactive Simulacra of Human Behavior."](https://arxiv.org/abs/2304.03442) 奠基性论文，演示了在模拟智能体中做新近度加权的记忆检索。

- Zhong, W., et al. (2024). ["MemoryBank: Enhancing Large Language Models with Long-Term Memory."](https://arxiv.org/abs/2305.10250) 提出受艾宾浩斯遗忘曲线启发的显式遗忘机制，应用于 LLM 记忆。

- Tulving, E. (2002). ["Episodic Memory: From Mind to Brain."](https://doi.org/10.1146/annurev.psych.53.100901.135114) *Annual Review of Psychology*, 53, 1-25。关于带时间戳的人类记忆的认知科学奠基之作。

- Hassabis, D., & Maguire, E. A. (2007). ["Deconstructing Episodic Memory with Construction."](https://doi.org/10.1016/j.tics.2007.05.001) *Trends in Cognitive Sciences*, 11(7)。人类如何构建情景记忆，以及对人工记忆的启示。

- [OpenAI Embeddings 指南](https://platform.openai.com/docs/guides/embeddings?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)。本 notebook 所用嵌入模型的参考文档。

---

*← 上一篇：[17：记忆路由](../17_memory_routing/) · 下一篇：[19：遗忘与衰减](../19_forgetting_and_decay/) →*

## 🧪 动手练习

三个小挑战，加深你的理解。每个大约需要 10-30 分钟。

### 挑战 1：对比衰减函数
用 `TemporalMemoryStore` 跑同样的 10 个查询，跑两次：一次在 `TemporalDecay` 中用指数衰减，一次用线性衰减。对比每个查询的前 5 条结果，数一数有多少结果不同，并判断哪种衰减函数更偏向新记忆、哪种更偏向旧记忆。

### 挑战 2：半衰期调参
测试 1 小时、6 小时、24 小时和 7 天这几个半衰期取值。对每个取值，用 10 个时间敏感的问题（正确答案随时间变化）运行 `query()`。记录每种设置的准确率，找出在新旧信息之间取得最佳平衡的半衰期。

### 挑战 3：时序知识图谱
把 `TemporalMemoryStore` 和 08 知识图谱记忆中的三元组抽取结合起来。给每个三元组加上 `event_time`，让关系带上时间戳。实现一个 `query_as_of()`，返回某个时间点的图谱状态。用一段事实会随时间变化的对话来测试。

![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--18-temporal-memory--temporal-memory)